# Posterior Predictive Checks for Cosmological Parameter Inference

This notebook demonstrates how to perform posterior predictive checks to assess goodness-of-fit from cosmological parameter inference. 

Posterior predictive checks work by:
1. For each posterior sample θ, compute the test statistic T_obs for the observed data
2. For each posterior sample θ, generate replicated data from the model and compute T_rep
3. Calculate the p-value as the fraction of times T_rep > T_obs

A p-value close to 0.5 indicates good model fit, while values close to 0 or 1 suggest the model may not adequately describe the data.

**Reference:** https://mc-stan.org/docs/stan-users-guide/posterior-predictive-checks.html

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

# Set random seed for reproducibility
np.random.seed(42)

## Core Functions

These are the main functions for computing posterior predictive p-values and visualizing results.

In [ ]:
def posterior_predictive_pvalue(data, model_func, cov, theta_samples):
    """
    Calculate posterior predictive p-value.
    
    Parameters
    ----------
    data : np.ndarray
        Observed data vector of shape (n_features,)
    model_func : callable
        Function that takes theta parameters and returns model prediction
    cov : np.ndarray
        Covariance matrix of shape (n_features, n_features)
    theta_samples : np.ndarray
        Posterior samples of parameters, shape (n_samples, n_params)
    
    Returns
    -------
    pval : float
        Posterior predictive p-value
    T_obs : np.ndarray
        Array of chi-squared statistics for observed data
    T_rep : np.ndarray
        Array of chi-squared statistics for replicated data
    """
    Cinv = np.linalg.inv(cov)
    T_obs, T_rep = [], []

    for theta in theta_samples:
        model = model_func(theta)
        d_rep = np.random.multivariate_normal(model, cov)
        T_obs.append((data - model) @ Cinv @ (data - model))
        T_rep.append((d_rep - model) @ Cinv @ (d_rep - model))

    T_obs = np.array(T_obs)
    T_rep = np.array(T_rep)
    pval = np.mean(T_rep > T_obs)
    
    return pval, T_obs, T_rep


def compute_chi2_at_best_fit(data, model_func, cov, best_fit_params):
    """
    Compute chi-squared and chi-squared per degree of freedom at best fit.
    
    Parameters
    ----------
    data : np.ndarray
        Observed data vector
    model_func : callable
        Model function
    cov : np.ndarray
        Covariance matrix
    best_fit_params : np.ndarray
        Best fit parameter values
    
    Returns
    -------
    chi2 : float
        Chi-squared value at best fit
    dof : int
        Degrees of freedom
    chi2_per_dof : float
        Chi-squared per degree of freedom
    """
    model = model_func(best_fit_params)
    residual = data - model
    Cinv = np.linalg.inv(cov)
    chi2 = residual @ Cinv @ residual
    
    n_data = len(data)
    n_params = len(best_fit_params)
    dof = n_data - n_params
    chi2_per_dof = chi2 / dof
    
    return chi2, dof, chi2_per_dof

## Visualization Functions

In [ ]:
def plot_ppc_summary(T_obs, T_rep, pval, chi2_at_best, dof, chi2_per_dof, title_suffix=""):
    """
    Create comprehensive summary plot with multiple panels.
    """
    fig = plt.figure(figsize=(15, 10))
    gs = gridspec.GridSpec(2, 2, figure=fig, hspace=0.3, wspace=0.3)
    
    # Panel 1: Histogram comparison
    ax1 = fig.add_subplot(gs[0, 0])
    
    # Use appropriate binning based on data range
    bins_T_obs = np.histogram_bin_edges(T_obs, bins='auto')
    bins_T_rep = np.histogram_bin_edges(T_rep, bins='auto')
    
    ax1.hist(T_rep, bins=bins_T_rep, alpha=0.6, label='T_rep (replicated)', 
             color='steelblue', edgecolor='black', linewidth=0.5, density=False)
    ax1.hist(T_obs, bins=bins_T_obs, alpha=0.6, label='T_obs (observed)', 
             color='coral', edgecolor='black', linewidth=0.5, density=False)
    ax1.axvline(np.mean(T_rep), color='steelblue', linestyle='--', linewidth=2, 
                label=f'Mean T_rep = {np.mean(T_rep):.1f}')
    ax1.axvline(np.mean(T_obs), color='coral', linestyle='--', linewidth=2,
                label=f'Mean T_obs = {np.mean(T_obs):.1f}')
    ax1.set_xlabel('Chi-squared statistic', fontsize=11)
    ax1.set_ylabel('Count', fontsize=11)
    ax1.set_title('Distribution of Test Statistics', fontsize=12, fontweight='bold')
    ax1.legend(fontsize=9)
    ax1.grid(True, alpha=0.3)
    
    # Panel 2: Scatter plot
    ax2 = fig.add_subplot(gs[0, 1])
    ax2.scatter(T_obs, T_rep, alpha=0.5, s=20, color='steelblue', 
                edgecolors='black', linewidth=0.5)
    
    # Add diagonal line
    all_vals = np.concatenate([T_obs, T_rep])
    min_val, max_val = all_vals.min(), all_vals.max()
    ax2.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, 
             alpha=0.7, label='T_rep = T_obs')
    ax2.set_xlabel('T_obs', fontsize=11)
    ax2.set_ylabel('T_rep', fontsize=11)
    ax2.set_title('T_obs vs T_rep', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9)
    ax2.grid(True, alpha=0.3)
    
    # Panel 3: Cumulative distribution
    ax3 = fig.add_subplot(gs[1, 0])
    sorted_T_obs = np.sort(T_obs)
    sorted_T_rep = np.sort(T_rep)
    ax3.plot(sorted_T_obs, np.arange(1, len(T_obs) + 1) / len(T_obs), 
             label='T_obs', color='coral', linewidth=2)
    ax3.plot(sorted_T_rep, np.arange(1, len(T_rep) + 1) / len(T_rep), 
             label='T_rep', color='steelblue', linewidth=2)
    ax3.set_xlabel('Chi-squared statistic', fontsize=11)
    ax3.set_ylabel('Cumulative probability', fontsize=11)
    ax3.set_title('Cumulative Distribution Functions', fontsize=12, fontweight='bold')
    ax3.legend(fontsize=9)
    ax3.grid(True, alpha=0.3)
    
    # Panel 4: Summary statistics
    ax4 = fig.add_subplot(gs[1, 1])
    ax4.axis('off')
    
    fit_status = 'GOOD FIT ✓' if 0.05 < pval < 0.95 else 'POOR FIT ✗'
    
    summary_text = f"""
    POSTERIOR PREDICTIVE CHECK SUMMARY
    {'=' * 42}
    
    P-value:              {pval:.4f}
    Model fit status:     {fit_status}
    
    {'─' * 42}
    
    T_obs (observed):
      Mean:               {np.mean(T_obs):.2f}
      Std:                {np.std(T_obs):.2f}
      Min:                {np.min(T_obs):.2f}
      Max:                {np.max(T_obs):.2f}
    
    T_rep (replicated):
      Mean:               {np.mean(T_rep):.2f}
      Std:                {np.std(T_rep):.2f}
      Min:                {np.min(T_rep):.2f}
      Max:                {np.max(T_rep):.2f}
    
    {'─' * 42}
    
    Chi-squared at best fit: {chi2_at_best:.2f}
    Degrees of freedom:      {dof}
    Chi-squared per DOF:     {chi2_per_dof:.4f}
    
    {'─' * 42}
    
    Samples used:            {len(T_obs)}
    Points where T_rep > T_obs: {np.sum(T_rep > T_obs)} ({100*pval:.1f}%)
    """
    
    ax4.text(0.1, 0.95, summary_text, transform=ax4.transAxes, 
             fontsize=10, verticalalignment='top', 
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
    
    # Add colored box for fit status
    status_y = 0.78
    if 0.05 < pval < 0.95:
        ax4.add_patch(plt.Rectangle((0.58, status_y), 0.3, 0.05, 
                                     transform=ax4.transAxes, 
                                     facecolor='lightgreen', alpha=0.5))
    else:
        ax4.add_patch(plt.Rectangle((0.58, status_y), 0.3, 0.05, 
                                     transform=ax4.transAxes, 
                                     facecolor='lightcoral', alpha=0.5))
    
    fig.suptitle(f'Posterior Predictive Check Results{title_suffix}', 
                 fontsize=16, fontweight='bold', y=0.98)
    
    plt.tight_layout()
    return fig

## Example 1: Well-Fitting Model

This example demonstrates posterior predictive checks with a well-specified model. We create synthetic data from a quadratic model and fit it with the correct model form.

In [ ]:
# Set random seed
np.random.seed(42)

# Define dimensions
n_features = 20  # Number of data points
n_params = 3     # Number of model parameters
n_samples = 500  # Number of posterior samples

print("Example 1: Well-Fitting Model")
print("=" * 60)
print(f"Number of data points: {n_features}")
print(f"Number of parameters: {n_params}")
print(f"Number of posterior samples: {n_samples}")

# True parameter values
theta_true = np.array([1.0, 2.0, -0.5])
print(f"\nTrue parameters: {theta_true}")

# Create design matrix for quadratic model
x = np.linspace(0, 10, n_features)
X = np.vstack([np.ones(n_features), x, x**2]).T

# Generate observed data with noise
y_true = X @ theta_true
noise_std = 0.5
noise = np.random.normal(0, noise_std, n_features)
data = y_true + noise

# Create covariance matrix (assuming uncorrelated Gaussian noise)
cov = noise_std**2 * np.eye(n_features)

# Define model function
def model_func(theta):
    """Quadratic model: y = theta[0] + theta[1]*x + theta[2]*x^2"""
    return X @ theta

# Generate posterior samples (simulating MCMC output)
# Add some spread around true values
posterior_std = np.array([0.1, 0.15, 0.08])
theta_samples = theta_true + np.random.randn(n_samples, n_params) * posterior_std

print(f"\nPosterior samples:")
print(f"  Mean: {np.mean(theta_samples, axis=0)}")
print(f"  Std:  {np.std(theta_samples, axis=0)}")

In [ ]:
# Compute posterior predictive p-value
print("\nComputing posterior predictive p-value...")
pval1, T_obs1, T_rep1 = posterior_predictive_pvalue(data, model_func, cov, theta_samples)

print(f"\nResults:")
print(f"  Posterior predictive p-value: {pval1:.4f}")
print(f"  Mean T_obs (observed chi-squared): {np.mean(T_obs1):.2f}")
print(f"  Mean T_rep (replicated chi-squared): {np.mean(T_rep1):.2f}")
print(f"  Std T_obs: {np.std(T_obs1):.2f}")
print(f"  Std T_rep: {np.std(T_rep1):.2f}")

# Interpretation
print("\nInterpretation:")
if 0.05 < pval1 < 0.95:
    print("  ✓ P-value is in reasonable range (0.05 - 0.95)")
    print("  ✓ Model appears to fit the data well")
else:
    print("  ✗ P-value is outside reasonable range")
    print("  ✗ Model may not fit the data well")

In [ ]:
# Compute chi-squared at best fit
best_fit1 = np.mean(theta_samples, axis=0)
chi2_1, dof1, chi2_per_dof1 = compute_chi2_at_best_fit(data, model_func, cov, best_fit1)

print(f"\nChi-squared at best fit (posterior mean):")
print(f"  Chi-squared: {chi2_1:.2f}")
print(f"  Degrees of freedom: {dof1}")
print(f"  Chi-squared per DOF: {chi2_per_dof1:.4f}")

if 0.5 < chi2_per_dof1 < 2.0:
    print("  ✓ Chi-squared per DOF is reasonable")
else:
    print("  ! Chi-squared per DOF may indicate issues")

In [ ]:
# Generate summary plot
fig1 = plot_ppc_summary(T_obs1, T_rep1, pval1, chi2_1, dof1, chi2_per_dof1, 
                        title_suffix=" - Well-Fitting Model")
plt.show()

## Example 2: Misspecified Model (Poor Fit)

This example demonstrates posterior predictive checks when the model is misspecified. We create data from a quadratic model but try to fit it with only a linear model.

In [ ]:
# Set random seed
np.random.seed(123)

n_features2 = 20
n_params2 = 2  # Using only 2 params when true model needs 3
n_samples2 = 500

print("Example 2: Misspecified Model (Poor Fit)")
print("=" * 60)
print(f"Number of data points: {n_features2}")
print(f"Number of parameters: {n_params2} (model is misspecified!)")
print(f"Number of posterior samples: {n_samples2}")

# True model is quadratic, but we'll fit with only linear
x2 = np.linspace(0, 10, n_features2)
X_true2 = np.vstack([np.ones(n_features2), x2, x2**2]).T
theta_true2 = np.array([1.0, 2.0, -0.5])

y_true2 = X_true2 @ theta_true2
noise_std2 = 0.5
noise2 = np.random.normal(0, noise_std2, n_features2)
data2 = y_true2 + noise2

cov2 = noise_std2**2 * np.eye(n_features2)

# Define WRONG model (linear instead of quadratic)
X_wrong = np.vstack([np.ones(n_features2), x2]).T

def model_func_wrong(theta):
    """Linear model (misspecified - should be quadratic)"""
    return X_wrong @ theta

# Generate posterior samples for wrong model
theta_samples_wrong = np.random.randn(n_samples2, n_params2) * 0.1 + np.array([5.0, 0.5])

print(f"\nPosterior samples (misspecified model):")
print(f"  Mean: {np.mean(theta_samples_wrong, axis=0)}")
print(f"  Std:  {np.std(theta_samples_wrong, axis=0)}")

In [ ]:
# Compute posterior predictive p-value for misspecified model
print("\nComputing posterior predictive p-value...")
pval2, T_obs2, T_rep2 = posterior_predictive_pvalue(data2, model_func_wrong, cov2, theta_samples_wrong)

print(f"\nResults with misspecified model:")
print(f"  Posterior predictive p-value: {pval2:.4f}")
print(f"  Mean T_obs: {np.mean(T_obs2):.2f}")
print(f"  Mean T_rep: {np.mean(T_rep2):.2f}")
print(f"  Std T_obs: {np.std(T_obs2):.2f}")
print(f"  Std T_rep: {np.std(T_rep2):.2f}")

print("\nInterpretation:")
if pval2 < 0.05 or pval2 > 0.95:
    print("  ✓ P-value correctly indicates poor model fit!")
    print("  ✓ Posterior predictive check detected model misspecification")
else:
    print("  ! P-value did not detect misspecification (may need more samples)")

In [ ]:
# Compute chi-squared at best fit
best_fit2 = np.mean(theta_samples_wrong, axis=0)
chi2_2, dof2, chi2_per_dof2 = compute_chi2_at_best_fit(data2, model_func_wrong, cov2, best_fit2)

print(f"\nChi-squared at best fit (posterior mean):")
print(f"  Chi-squared: {chi2_2:.2f}")
print(f"  Degrees of freedom: {dof2}")
print(f"  Chi-squared per DOF: {chi2_per_dof2:.4f}")

if chi2_per_dof2 > 2.0:
    print("  ✓ Chi-squared per DOF also indicates poor fit")

In [ ]:
# Generate summary plot
fig2 = plot_ppc_summary(T_obs2, T_rep2, pval2, chi2_2, dof2, chi2_per_dof2,
                        title_suffix=" - Misspecified Model")
plt.show()

## Summary and Interpretation

### What to Look For:

1. **P-value**: 
   - Values close to 0.5 indicate good fit
   - Values < 0.05 or > 0.95 suggest model misspecification

2. **Histogram**: 
   - T_obs and T_rep distributions should overlap significantly for good fit
   - Large separation indicates poor fit

3. **Scatter Plot**:
   - Points should scatter around the diagonal line (T_rep = T_obs)
   - Systematic deviation indicates model issues

4. **Chi-squared per DOF**:
   - Values around 1.0 indicate good fit
   - Values >> 1.0 indicate poor fit

### Using with Real Data:

To use these functions with your own cosmological inference results:

1. Load your data vector and covariance matrix
2. Define a model function that takes parameter samples and returns predictions
3. Load your MCMC posterior samples (e.g., from sunbird chains)
4. Call `posterior_predictive_pvalue()` with your data
5. Visualize results using `plot_ppc_summary()`

Example code structure:
```python
# Load your data
data = load_your_data()
cov = load_your_covariance()

# Define model
def my_model(theta):
    # Your emulator or model prediction
    return prediction

# Load chains
from sunbird.inference.samples import Chain
chain = Chain.load('chain.npy')
samples = chain.to_getdist(add_derived=False)
theta_samples = samples.samples

# Compute p-value
pval, T_obs, T_rep = posterior_predictive_pvalue(data, my_model, cov, theta_samples)

# Visualize
best_fit = np.mean(theta_samples, axis=0)
chi2, dof, chi2_per_dof = compute_chi2_at_best_fit(data, my_model, cov, best_fit)
plot_ppc_summary(T_obs, T_rep, pval, chi2, dof, chi2_per_dof)
```